# Member 2: Empirical MDP, policy iteration and behavior TD baseline

Run Member 1 first. All results here describe the synthetic training MDP, not clinical performance.

In [ ]:
import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, display

root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'main_pipeline.py').is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project root or a member folder")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from main_pipeline import load_configs, prepare_data, train_model, evaluate_model

output = Path(os.environ.get('ASS2_OUTPUT_DIR', root)).resolve()
print("Source: synthetic coursework demo; no clinical efficacy claim.")
print("Output:", output)


## 1. Train using the saved run configuration

Policy evaluation must converge before policy improvement is accepted. Hitting iteration limits raises an error.

In [ ]:
trainer = train_model(output)
mdp, pi_solver, td_solver = trainer.mdp, trainer.pi, trainer.td_eval
tr_train = pd.read_csv(output / 'data/trajectories_train.csv')
print("Converged:", pi_solver.converged, "| Iterations:", len(pi_solver.history))
pd.DataFrame(pi_solver.history)


## 2. Inspect empirical action support

Actions below the count threshold are excluded when alternatives exist. If a state has no action meeting the threshold, the most frequent observed action is retained explicitly as a fallback. An unobserved cluster has a placeholder self-loop; it is not evidence of a safe action.

In [ ]:
print("Transition tensor:", mdp.transitions.shape)
print("Below-threshold fallback states:", mdp.fallback_states)
print("Unobserved placeholder states:", mdp.unobserved_states)
assert np.allclose(mdp.transitions[mdp.valid_action_mask].sum(axis=1), 1)
pd.DataFrame({'state': range(mdp.n_states), 'allowed_actions': mdp.valid_action_mask[:mdp.n_states].sum(axis=1)})


## 3. Batch empirical-policy TD(0)

Each iteration averages TD targets for observed state-action pairs. State values weight Q by empirical action frequencies, not uniformly over 25 actions. Training-state means below are not held-out OPE estimates.

In [ ]:
print("TD iterations:", td_solver.iterations)
pd.DataFrame({'AI training-state V': pi_solver.V[:mdp.n_states], 'Behavior training-state V': td_solver.get_clinician_state_values()}).describe()


## 4. Policy iteration convergence

In [ ]:
import matplotlib.pyplot as plt
history = pd.DataFrame(pi_solver.history)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history['iteration'], history['mean_v'], marker='o')
axes[0].set(xlabel='Iteration', ylabel='Mean training-state V')
axes[1].bar(history['iteration'], history['actions_changed'])
axes[1].set(xlabel='Iteration', ylabel='Changed actions')
fig.suptitle('Synthetic empirical MDP')
fig.tight_layout()
plt.show()
plt.close(fig)


## 5. Inspect actions and export metadata

In [ ]:
policy = trainer.get_policy()
display(pd.DataFrame({'state': range(len(policy)), 'action': policy, 'fluid_bin': policy // 5, 'vaso_bin': policy % 5}).head(15))
metadata = json.loads((output / 'artifacts/ai_policy.json').read_text())
display({key: value for key, value in metadata.items() if key != 'policy'})
